# Exercise 10 — Reasoning Frameworks: The Game of 24

In week 7, **chain-of-thought** (CoT) prompting made the model much better at math word problems: by writing out intermediate steps, the model spends more computation on the problem, and every step can build on the previous ones. But a chain of thought is still generated **left to right, in a single pass**. Once the model has written a step, it is committed to it: if an early step was a bad choice, there is no going back. That is fine for a word problem with one obvious path to the answer, but not for problems that need **search**: trying an option, noticing that it leads nowhere, and trying another one.

In this notebook you test this on the **Game of 24**, a small puzzle whose answers are easy to check but hard to find, and you implement two frameworks that let a model explore instead of committing to its first idea:

- **ReAct** ([Yao et al., 2023a](https://arxiv.org/abs/2210.03629)): the model alternates *reasoning* with *actions*, and an environment (your code) executes every action and reports what happened.
- **Tree of Thoughts** ([Yao et al., 2023b](https://arxiv.org/abs/2305.10601)): the model proposes several possible next steps and evaluates how promising they are, and a search algorithm decides which ones to pursue.

## What you will do
1. **The Game of 24**: the puzzles, checking an answer exactly, and a classical search that solves every puzzle.
2. **Setup**: the Groq API, as in week 7.
3. **Baselines**: few-shot prompting and chain-of-thought.
4. **ReAct**: one step at a time, with feedback from an environment.
5. **Tree of Thoughts**: proposing, evaluating and searching over partial solutions.
6. **Comparison**: accuracy against cost.
7. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook calls an online API**, the same Groq API as in week 7, so you need your free Groq API key. No GPU is needed and nothing is downloaded: Part 1 runs entirely on your computer. Tree of Thoughts makes many requests per puzzle, so we keep the experiments small. In total they use roughly 110,000 tokens, a bit more than half of the free daily budget (as of October 2026: 8,000 tokens per minute and 200,000 tokens per day for the model we use), and take about 15 minutes because of the per-minute limit. The client waits and retries automatically when it hits the limit. Every reply is cached in `groq_cache.json` next to the notebook, so running a cell a second time costs nothing.

In [ ]:
import ast
import hashlib
import json
import operator
import os
import re
import time
from fractions import Fraction
from getpass import getpass

import pandas as pd
from groq import Groq

## 1. The Game of 24

You get **four numbers** and have to combine them with **+, −, ×, ÷** and brackets to make **24**, using every number exactly once. For example, `4 7 8 8` can be solved with `(7 - 8 / 8) * 4 = 24`.

The dataset contains the 1,362 puzzles of [4nums.com](https://www.4nums.com/game/difficulties/), all of which can be solved, sorted by how long people took to solve them: `Rank` 1 is the easiest puzzle, and the last ones are solved by fewer than a third of the players (`Solved rate`). Like Yao et al. (2023b), we test on the puzzles with ranks 901–1000, which are relatively hard.

In [ ]:
df = pd.read_csv("dataset_game24.csv.gz")
print(f"{len(df)} puzzles")
display(df.head())
display(df.iloc[900:905])

**1.1 Play the game.** Solve these puzzles by hand: `1 1 4 6` (rank 1), `1 1 11 11` (rank 2) and `4 5 6 10` (rank 901). How did you search for a solution: did you try steps in a fixed order, or did you have a feeling for which steps were promising? Did you have to undo a step?

---

*Answer:*

- `1 1 4 6`: `1 * 1 * 4 * 6 = 24`
- `1 1 11 11`: `11 + 11 + 1 + 1 = 24`
- `4 5 6 10`: `(10 - 6) * 5 + 4 = 24`

Most people do not try steps in a fixed order. They use **heuristics**: 24 = 4 × 6 = 3 × 8 = 2 × 12, so they look for ways to make a 6 when they have a 4, or a 3 when they have an 8; and they quickly see that a state like `1 2 3` is "too small" to ever reach 24. When a promising idea does not work out, they **go back** and try something else. These are exactly the two ingredients of Tree of Thoughts in Part 5: an evaluation of how promising a partial solution is, and a search that can backtrack.

---

### Checking an answer exactly

An answer is correct if it uses exactly the four numbers of the puzzle, each once, and equals 24. Checking the value with floating-point numbers is not reliable, because divisions produce fractions:

In [ ]:
print(8 / (3 - 8 / 3))
print(8 / (3 - 8 / 3) == 24)

So we compute with **exact fractions** (`fractions.Fraction`) instead. The function `evaluate_expression` below computes the value of an arithmetic expression exactly. It does not use Python's `eval`, which would run any code a model writes, but walks through the expression's syntax tree and only allows numbers, `+ - * /` and brackets.

In [ ]:
OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}


def evaluate_expression(expression):
    # The exact value of an expression with numbers, + - * / and brackets, as a Fraction.
    # Raises a ValueError for anything else, and a ZeroDivisionError for a division by zero.
    def value(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return Fraction(str(node.value))
        if isinstance(node, ast.BinOp) and type(node.op) in OPERATIONS:
            return OPERATIONS[type(node.op)](value(node.left), value(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -value(node.operand)
        raise ValueError(f"only numbers, + - * / and brackets are allowed, not {type(node).__name__}")

    expression = expression.replace("×", "*").replace("÷", "/").replace("−", "-")
    try:
        tree = ast.parse(expression.strip(), mode="eval")
    except SyntaxError:
        raise ValueError(f"not a valid expression: {expression!r}")
    return value(tree.body)


print(evaluate_expression("8 / (3 - 8 / 3)"))
print(evaluate_expression("(4 + 8) * (6 - 4)"))

**1.2 Complete `check_answer`.** It gets an expression (possibly followed by `= 24`) and a puzzle such as `"4 5 6 10"`, and returns `True` if the expression uses exactly the numbers of the puzzle, each once, and equals 24. For expressions that cannot be evaluated it must return `False`, not crash: later, the expressions will come from a language model.

Hints:
- `expression.split("=")[0]` removes a trailing `= 24`.
- `re.findall(r"\d+", text)` finds all numbers in a text. Compare the *sorted* lists of the numbers (as `int`s) in the expression and in the puzzle.
- `evaluate_expression` raises a `ValueError` for an invalid expression and a `ZeroDivisionError` for a division by zero.

In [ ]:
def check_answer(expression, puzzle):
    # True if the expression uses every number of the puzzle exactly once and equals 24
    expression = expression.split("=")[0]
    if sorted(int(n) for n in re.findall(r"\d+", expression)) != sorted(int(n) for n in puzzle.split()):
        return False
    try:
        return evaluate_expression(expression) == 24
    except (ValueError, ZeroDivisionError):
        return False

In [ ]:
# ✅ Check check_answer
assert check_answer("(10 - 6) * 5 + 4", "4 5 6 10") is True
assert check_answer("(10 - 6) * 5 + 4 = 24", "4 5 6 10") is True, "a trailing '= 24' should be ignored"
assert check_answer("8 / (3 - 8 / 3)", "3 3 8 8") is True, "compare exact fractions, not floats"
assert check_answer("10 + 6 + 5 + 4", "4 5 6 10") is False, "the expression must equal 24"
assert check_answer("(10 - 6) * 6", "4 5 6 10") is False, "the expression must use every number of the puzzle"
assert check_answer("5 * 4 + 4 + 6 - 10 + 4", "4 5 6 10") is False, "every number may be used only once"
assert check_answer("6 * 4 + 5 / (5 - 5)", "4 5 5 6") is False, "a division by zero is not an answer"
assert check_answer("abs(4 - 5) * 6 * 10", "4 5 6 10") is False, "only + - * / and brackets are allowed"
assert check_answer("", "4 5 6 10") is False
print("Looks good ✅")

### States and steps

A solution is a sequence of three **steps**. Each step takes two of the numbers that are left and replaces them by their sum, difference, product or quotient:

```
4 5 6 10   --(10 - 6 = 4)-->   4 4 5   --(4 * 5 = 20)-->   4 20   --(20 + 4 = 24)-->   24
```

We call the numbers that are left the **state**. To be able to write down the solution at the end, we store for every number the expression that produced it: a state is a list of `(value, expression)` pairs. Three helpers are given:
- `initial_state(puzzle)` turns a puzzle into a state,
- `show(state)` writes a state the way the model will see it: `"4 4 5"`, with fractions rounded to two decimals,
- `combine(state, i, op, j)` combines the numbers at positions `i` and `j` with the operation `op` and returns the step as text and the new state.

In [ ]:
APPLY = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


def initial_state(puzzle):
    # A state is a list of (value, expression) pairs: the numbers that are left, and how each was made
    return [(Fraction(int(n)), n) for n in puzzle.split()]


def fmt(value):
    # A number the way the model sees it: 7, -2 or 2.67
    return str(value.numerator) if value.denominator == 1 else f"{float(value):.2f}".rstrip("0")


def show(state):
    # The numbers that are left, smallest first, e.g. "4 4 5"
    return " ".join(fmt(value) for value, _ in sorted(state, key=lambda pair: pair[0]))


def combine(state, i, op, j):
    # Combine the numbers at positions i and j with op ("+", "-", "*" or "/"); returns (step, new_state)
    (a, expression_a), (b, expression_b) = state[i], state[j]
    value = APPLY[op](a, b)  # a ZeroDivisionError if op is "/" and b is 0
    rest = [pair for k, pair in enumerate(state) if k not in (i, j)]
    new_state = rest + [(value, f"({expression_a} {op} {expression_b})")]
    return f"{fmt(a)} {op} {fmt(b)} = {fmt(value)} (left: {show(new_state)})", new_state


state = initial_state("4 5 6 10")
step, new_state = combine(state, 3, "-", 2)
print(step)
print(new_state)

**1.3 Complete `next_states`**, which returns every `(step, new_state)` pair that can be reached from a state in one step. Addition and multiplication are commutative, so use them only once for each pair of positions (`i < j`); subtraction and division are not, so use them in both orders (`i != j`). Skip divisions by zero.

In [ ]:
def next_states(state):
    # Every (step, new_state) pair that can be reached from state in one step
    steps = []
    for i in range(len(state)):
        for j in range(len(state)):
            if i < j:
                steps += [combine(state, i, "+", j), combine(state, i, "*", j)]
            if i != j:
                steps.append(combine(state, i, "-", j))
                if state[j][0] != 0:
                    steps.append(combine(state, i, "/", j))
    return steps

In [ ]:
# ✅ Check next_states
assert len(next_states(initial_state("1 2"))) == 6, "two numbers can be combined in 6 ways: a+b, a*b, a-b, b-a, a/b and b/a"
assert len(next_states(initial_state("4 5 6 10"))) == 36, "four numbers: 6 pairs, 6 ways each"
assert len(next_states(initial_state("0 3"))) == 5, "dividing by 0 is not allowed"
assert {show(s) for _, s in next_states(initial_state("2 3"))} == {"5", "6", "-1", "1", "0.67", "1.5"}
print("Looks good ✅")

**1.4 Complete `solve`**, a **depth-first search** over the steps: if one number is left, return its expression if it equals 24, and `None` otherwise. If more numbers are left, try every next state in turn, and return the first solution found from it.

In [ ]:
def solve(state):
    # Depth-first search: an expression for 24 that uses the numbers in state, or None
    if len(state) == 1:
        value, expression = state[0]
        return expression if value == 24 else None
    for _, new_state in next_states(state):
        expression = solve(new_state)
        if expression is not None:
            return expression
    return None

In [ ]:
# ✅ Check solve
solution = solve(initial_state("4 5 6 10"))
print(solution)
assert solution is not None and check_answer(solution, "4 5 6 10")
assert solve(initial_state("1 1 1 1")) is None, "1 1 1 1 has no solution"
print("Looks good ✅")

The cell below solves every puzzle in the dataset.

In [ ]:
start = time.time()
solutions = [solve(initial_state(puzzle)) for puzzle in df["Puzzles"]]
print(f"Solved {sum(s is not None for s in solutions)} of {len(df)} puzzles in {time.time() - start:.1f} seconds")
print(f"The hardest puzzle, {df['Puzzles'].iloc[-1]}: {solutions[-1]}")

**1.5 Why use a language model at all?** From four numbers, there are at most 36 × 18 × 6 = 3,888 ways to play the three steps, and a computer tries them all in a few milliseconds. So why is the Game of 24 still an interesting test for language models? Think of what the programmer of `solve` had to know about the problem, and of problems where the possible next steps cannot be listed.

---

*Answer:*

The Game of 24 is a **controlled testbed**: answers can be checked exactly, a solution needs several dependent steps, and it needs search, because only a few of the 3,888 paths reach 24. Since we know the ground truth (and can solve every puzzle), we can see exactly where a model goes wrong.

`solve` works because a programmer has formalised the problem: the possible moves, the goal, and how to check it. A language model gets the rules in plain words. For most real problems, such as writing a proof, a program, a plan or an essay, the "moves" are sentences or lines of code: there are astronomically many possible next steps, and nobody can list them, let alone try them all. There, the language model is useful *inside* the search: it **proposes** a few plausible next steps and **judges** how promising a partial solution is. That is exactly how Tree of Thoughts uses it, and the Game of 24 lets us study the idea on a problem where we can check every answer.

---

## 2. Setup

We use the same setup as in weeks 7 and 9: a 27 B parameter model on Groq, and a cached `chat` function. Run the cell and paste your API key when you are asked for it.

> **Keep your key secret.** Never write it into a code cell. If you set the environment variable `GROQ_API_KEY` before starting Jupyter, the cell does not ask at all.

As in week 7, `reasoning_effort="none"` switches off the model's built-in "thinking" mode, in which it reasons in hidden tokens before it answers: in this notebook, *we* decide how the model reasons.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

client = Groq(max_retries=8)  # reads GROQ_API_KEY; on a rate-limit error it waits and tries again (up to 8 times)

# A 27 B parameter open-weights model. If it is no longer available, pick another one from
# https://console.groq.com/docs/models and check its rate limits.
MODEL = "qwen/qwen3.8-27b"

CACHE_PATH = "groq_cache.json"
try:
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}


def chat(messages, temperature=0.0, max_tokens=256, sample=0):
    # Send a conversation to the model; returns the reply (a string) and the number of tokens used.
    key = hashlib.sha256(json.dumps([MODEL, messages, temperature, max_tokens, sample]).encode()).hexdigest()
    if key not in cache:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
        )
        cache[key] = {"reply": response.choices[0].message.content, "tokens": response.usage.total_tokens}
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f)
    return cache[key]["reply"], cache[key]["tokens"]

We test every method on the same `N_TEST = 10` puzzles (ranks 901–910), except Tree of Thoughts, which is much more expensive and only gets the first `N_TOT = 5` of them. `run_method` runs a method on a list of puzzles, checks every answer with your `check_answer` (we never trust the model's own claim that it found 24), and collects the results and the tokens in `results`.

In [ ]:
N_TEST, N_TOT = 10, 5
test_puzzles = list(df["Puzzles"][900 : 900 + N_TEST])
print(test_puzzles)

results = {}


def run_method(name, solve_fn, puzzles):
    # Run a method on the puzzles; solve_fn(puzzle) returns (expression or None, tokens used)
    rows = []
    for puzzle in puzzles:
        expression, tokens = solve_fn(puzzle)
        correct = bool(expression) and check_answer(expression, puzzle)
        rows.append({"puzzle": puzzle, "answer": expression, "correct": correct, "tokens": tokens})
        print(f"{puzzle:12} {'✓' if correct else '✗'}  {tokens:6,} tokens  {expression}")
    results[name] = rows
    solved = sum(row["correct"] for row in rows)
    print(f"{name}: solved {solved}/{len(rows)}, {sum(row['tokens'] for row in rows) / len(rows):,.0f} tokens per puzzle")

## 3. Baselines: few-shot prompting and chain-of-thought

We start with the two kinds of prompts from week 7. The **input-output (IO)** prompt of Yao et al. (2023b) shows five solved puzzles and asks for the answer directly:

In [ ]:
IO_PROMPT = """Use numbers and basic arithmetic operations (+ - * /) to obtain 24. Use every number exactly once. Reply with just the answer, in the format of the examples.

Input: 4 4 6 8
Answer: (4 + 8) * (6 - 4) = 24

Input: 2 9 10 12
Answer: 2 * 12 * (10 - 9) = 24

Input: 4 9 10 13
Answer: (13 - 9) * (10 - 4) = 24

Input: 1 4 8 8
Answer: (8 / 4 + 1) * 8 = 24

Input: 5 5 5 9
Answer: 5 + 5 + 5 + 9 = 24

Input: {input}
"""

print(IO_PROMPT.format(input=test_puzzles[0]))

**3.1 Complete `extract_answer`**, which returns the expression after the *last* `Answer:` in a reply, without the `= 24`, or `""` if the reply contains no answer. Models sometimes use Markdown, so also remove `*`, `` ` `` and spaces from both ends of the expression: `**Answer:** 4 * 6` should give `4 * 6`.

Hint: `re.findall(r"answer:(.*)", reply, flags=re.IGNORECASE)` finds the text after every `Answer:`, up to the end of its line. `text.strip(" *`")` removes these characters from both ends of `text`.

In [ ]:
def extract_answer(reply):
    # The expression after the last "Answer:" in the reply, without "= 24"; "" if there is none
    answers = re.findall(r"answer:(.*)", reply, flags=re.IGNORECASE)
    if not answers:
        return ""
    return answers[-1].split("=")[0].strip(" *`")

In [ ]:
# ✅ Check extract_answer
assert extract_answer("Answer: (4 + 8) * (6 - 4) = 24") == "(4 + 8) * (6 - 4)"
assert extract_answer("Steps:\n5 + 5 = 10 (left: 5 9 10)\nAnswer: 5 + 5 + 5 + 9 = 24\n") == "5 + 5 + 5 + 9"
assert extract_answer("answer: 2 * 12 * (10 - 9)") == "2 * 12 * (10 - 9)", "ignore upper and lower case"
assert extract_answer("**Answer:** 4 * 6 = 24") == "4 * 6", "remove the Markdown"
assert extract_answer("Answer: 1 + 1 = 2\nAnswer: 4 * 6 = 24") == "4 * 6", "take the last answer"
assert extract_answer("I cannot find a solution.") == ""
print("Looks good ✅")

Now run the IO prompt on the test puzzles.

In [ ]:
def io_solve(puzzle):
    reply, tokens = chat([{"role": "user", "content": IO_PROMPT.format(input=puzzle)}], max_tokens=64)
    return extract_answer(reply), tokens


run_method("IO", io_solve, test_puzzles)

**3.2 Write a chain-of-thought prompt.** The CoT prompt of Yao et al. (2023b) shows the three steps of every demonstration before its answer, in exactly the format of our steps: `a op b = c (left: ...)`. Complete `COT_PROMPT` with **at least two more demonstrations** in the same format. You can use the puzzles of the IO prompt, and let `solve` help you.

In [ ]:
COT_PROMPT = """Use numbers and basic arithmetic operations (+ - * /) to obtain 24. Each step, you are only allowed to choose two of the remaining numbers to obtain a new number. Use the format of the examples.

Input: 4 4 6 8
Steps:
4 + 8 = 12 (left: 4 6 12)
6 - 4 = 2 (left: 2 12)
2 * 12 = 24 (left: 24)
Answer: (6 - 4) * (4 + 8) = 24

Input: 2 9 10 12
Steps:
12 * 2 = 24 (left: 9 10 24)
10 - 9 = 1 (left: 1 24)
24 * 1 = 24 (left: 24)
Answer: (12 * 2) * (10 - 9) = 24

Input: 4 9 10 13
Steps:
13 - 10 = 3 (left: 3 4 9)
9 - 3 = 6 (left: 4 6)
4 * 6 = 24 (left: 24)
Answer: 4 * (9 - (13 - 10)) = 24

Input: 1 4 8 8
Steps:
8 / 4 = 2 (left: 1 2 8)
1 + 2 = 3 (left: 3 8)
3 * 8 = 24 (left: 24)
Answer: (1 + 8 / 4) * 8 = 24

Input: 5 5 5 9
Steps:
5 + 5 = 10 (left: 5 9 10)
10 + 5 = 15 (left: 9 15)
15 + 9 = 24 (left: 24)
Answer: ((5 + 5) + 5) + 9 = 24

Input: {input}
"""

In [ ]:
# ✅ Check COT_PROMPT
assert "TODO" not in COT_PROMPT and COT_PROMPT.strip().endswith("Input: {input}"), "the prompt should end with 'Input: {input}'"
demo_puzzles = re.findall(r"Input: (\d+ \d+ \d+ \d+)", COT_PROMPT)
demo_answers = [extract_answer(line) for line in re.findall(r"Answer:.*", COT_PROMPT)]
assert len(demo_puzzles) >= 3 and len(demo_answers) == len(demo_puzzles), "at least three demonstrations, each with one answer"
for puzzle, answer in zip(demo_puzzles, demo_answers):
    assert check_answer(answer, puzzle), f"the answer {answer!r} of the demonstration {puzzle!r} is not correct"
assert COT_PROMPT.count("(left: ") >= 3 * len(demo_puzzles), "every demonstration should show its three steps"
print("Looks good ✅")

In [ ]:
def cot_solve(puzzle):
    reply, tokens = chat([{"role": "user", "content": COT_PROMPT.format(input=puzzle)}], max_tokens=256)
    return extract_answer(reply), tokens


run_method("CoT", cot_solve, test_puzzles)

The cell below prints the chains of thought for the puzzles that CoT got wrong (the replies come from the cache).

In [ ]:
for row in results["CoT"]:
    if not row["correct"]:
        reply, _ = chat([{"role": "user", "content": COT_PROMPT.format(input=row["puzzle"])}], max_tokens=256)
        print(f"Input: {row['puzzle']}\n{reply}\n")

**3.3 What goes wrong?** Read the failed chains of thought and classify the mistakes: wrong arithmetic, a number that is used twice or not at all, a `(left: ...)` that does not match the step, an `Answer` that does not match the steps, or a chain of valid steps that simply ends at a number other than 24. Which of these mistakes could a program catch *while* the model is writing? And what would the model need to recover from the last kind?

---

*Answer:*

Typical mistakes: the arithmetic of a step is wrong (`9 * 3 = 24`), a `(left: ...)` list drops or duplicates a number, so that later steps use numbers that are no longer there, and, very often, a chain of valid steps ends at 22 or 30, after which the model writes an `Answer` that does not follow from its steps, or simply claims "= 24".

All mistakes except the last kind are **local**: a program can check every step as soon as it is written (is the arithmetic right, are the numbers available?) and report the problem, which is what the environment in Part 4 does. The last kind is a **bad decision**: every step was legal, but an early step made 24 unreachable. To recover, the model has to notice that the state is hopeless, **go back** and try a different step: it needs search. Yao et al. (2023b) found that about 60 % of the failed CoT samples were already doomed after the *first* step. With GPT-4, they report 7.3 % solved with IO, 4.0 % with CoT and 9.0 % with self-consistency over 100 chains of thought, against 74 % with Tree of Thoughts.

---

## 4. ReAct: reasoning and acting

ReAct (Yao et al., 2023a) turns the model into an **agent** that interacts with an **environment**. In every turn, the model writes a **Thought** (reasoning about what to do next) and an **Action**; the environment executes the action and replies with an **Observation**, which is added to the conversation before the next turn. In the paper, the actions were searches on Wikipedia. In the Game of 24, an action is a single step such as `10 - 6`, and the environment is your code:
- it checks that the step only uses numbers that are left, and computes the result itself, so that arithmetic and bookkeeping mistakes cannot go unnoticed,
- it tells the model which numbers are left,
- when the last number is not 24, it resets the numbers, so that the model can try again.

The environment has to read a step written by the model. Models write fractions as rounded decimals (`8 / 3 = 2.67`), so `find_number` matches a number with a tolerance of 0.01.

**4.1 Complete `apply_step`**, which returns the new state after a step, or raises a `ValueError` with a helpful message if the step uses a number that is not left (or the same number twice) or divides by zero.

Hint: `STEP_PATTERN.search(step)` finds the first `a op b` in a text, and `match.groups()` gives its three parts as strings. Find the position `i` of `a` with `find_number(state, a)`, and then the position of `b` with `find_number(state, b, skip=i)`, so that the same number is not used twice. Then use `combine`. We ignore the result that the model wrote down and compute it ourselves.

In [ ]:
STEP_PATTERN = re.compile(r"(-?\d+(?:\.\d+)?)\s*([-+*/×÷])\s*(-?\d+(?:\.\d+)?)")


def find_number(state, text, skip=None):
    # The position of a number in state that matches text (e.g. "2.67" matches 8/3), or None
    for k, (value, _) in enumerate(state):
        if k != skip and abs(float(value) - float(text)) < 0.01:
            return k
    return None


def apply_step(state, step):
    # The state after a step such as "10 - 6 = 4"; raises a ValueError if the step is not allowed
    match = STEP_PATTERN.search(step)
    if match is None:
        raise ValueError(f"could not find a step like '10 - 6' in {step!r}")
    a, op, b = match.groups()
    op = {"×": "*", "÷": "/"}.get(op, op)
    i = find_number(state, a)
    j = find_number(state, b, skip=i) if i is not None else None
    if i is None or j is None:
        raise ValueError(f"{a} {op} {b} uses a number that is not left")
    if op == "/" and state[j][0] == 0:
        raise ValueError("division by zero")
    return combine(state, i, op, j)[1]

In [ ]:
# ✅ Check apply_step
state = apply_step(initial_state("4 5 6 10"), "10 - 6 = 4 (left: 4 4 5)")
assert show(state) == "4 4 5", f"expected 4 4 5, got {show(state)}"
state = apply_step(state, "Action: 4 * 5")
assert show(state) == "4 20"
for bad_state, bad_step in [(state, "3 * 8"), (state, "20 / 20"), (state, "add the two numbers"), (initial_state("0 4"), "4 / 0")]:
    try:
        apply_step(bad_state, bad_step)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad_step!r} should raise a ValueError in the state {show(bad_state)}")
state = apply_step(state, "20 + 4 = 24")
assert show(state) == "24" and check_answer(state[0][1], "4 5 6 10"), "the expression of the last number should be a correct answer"
# Fractions, written as rounded decimals
state = apply_step(initial_state("3 3 8 8"), "8 / 3 = 2.67")
state = apply_step(state, "3 - 2.67 = 0.33")
state = apply_step(state, "8 / 0.33 = 24")
assert show(state) == "24" and check_answer(state[0][1], "3 3 8 8"), "compute with the exact values, not the rounded ones"
print("Looks good ✅")

The instructions for the model, with one example of the format:

In [ ]:
REACT_SYSTEM = """You are playing the Game of 24: combine the given numbers with + - * / to obtain 24. Each step combines two of the numbers that are left into a new number, so after three steps one number is left, and it must be 24.

In every turn, write exactly two lines:
Thought: <short reasoning about which two numbers to combine, and why>
Action: <number> <operation> <number>

After each action, you get an observation with the numbers that are left. If the last number is not 24, the numbers are reset and you can try again with a different first step.

Example:
Numbers: 4 9 10 13
Thought: 4 * 6 = 24, so I try to make 6 from 9, 10 and 13: 13 - 10 = 3 and 9 - 3 = 6.
Action: 13 - 10
Observation: left: 3 4 9
Thought: 9 - 3 = 6, and then 4 * 6 = 24.
Action: 9 - 3
Observation: left: 4 6
Thought: 4 * 6 = 24.
Action: 4 * 6
Observation: left: 24. Solved!"""

**4.2 Complete `react_solve`.** The conversation starts with the instructions and the puzzle. In every turn:
1. add the model's reply (a thought and an action) to the conversation as an `assistant` message,
2. apply the action (the text after the last `Action:`) with `apply_step`,
3. add an observation as a `user` message: the numbers that are left; or, if the step was not allowed, the error and the numbers that are still left; or, if the last number is not 24, that it is not 24 and that the numbers are reset to the puzzle (and reset them).

Return the expression as soon as the last number is 24, or `None` after `max_turns` turns, together with the tokens of all requests.

In [ ]:
def react_solve(puzzle, max_turns=6, verbose=False):
    # ReAct with our code as the environment; returns (expression or None, tokens used)
    state = initial_state(puzzle)
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user", "content": f"Numbers: {show(state)}"},
    ]
    tokens = 0
    for turn in range(max_turns):
        reply, used = chat(messages, max_tokens=150)
        tokens += used
        if verbose:
            print(reply)
        messages.append({"role": "assistant", "content": reply})
        action = reply.split("Action:")[-1]
        try:
            state = apply_step(state, action)
        except ValueError as error:
            observation = f"Observation: that step is not allowed: {error}. Numbers left: {show(state)}"
        else:
            if len(state) > 1:
                observation = f"Observation: left: {show(state)}"
            elif state[0][0] == 24:
                return state[0][1], tokens
            else:
                observation = f"Observation: left: {show(state)}, which is not 24. The numbers are reset to {puzzle}. Try a different first step."
                state = initial_state(puzzle)
        messages.append({"role": "user", "content": observation})
        if verbose:
            print(messages[-1]["content"], "\n")
    return None, tokens

In [ ]:
# ✅ Check react_solve, with a scripted "model" instead of the API
script = iter([
    "Thought: 10 + 6 = 16\nAction: 10 + 6",
    "Thought: 16 * 5 = 80\nAction: 16 * 5",
    "Thought: 80 - 4 = 76\nAction: 80 - 4",  # 76 is not 24: the numbers should be reset
    "Thought: 7 * 3 = 21\nAction: 7 * 3",    # not allowed: there is no 7
    "Thought: 10 - 6 = 4\nAction: 10 - 6",
    "Thought: 4 * 5 = 20\nAction: 4 * 5",
    "Thought: 20 + 4 = 24\nAction: 20 + 4",
])
sent = []


def scripted_chat(messages, **kwargs):
    sent.append(messages[-1]["content"])
    return next(script), 10


real_chat, chat = chat, scripted_chat
try:
    expression, tokens = react_solve("4 5 6 10", max_turns=10)
finally:
    chat = real_chat
assert expression is not None and check_answer(expression, "4 5 6 10"), "return the expression as soon as the last number is 24"
assert tokens == 70, "add up the tokens of all requests"
assert "4 5 16" in sent[1], "after a step, tell the model which numbers are left"
assert "4 5 6 10" in sent[3], "if the last number is not 24, reset the numbers to the puzzle"
assert "4 5 6 10" in sent[4], "if a step is not allowed, tell the model which numbers are still left"
print("Looks good ✅")

Watch the agent on one puzzle, then run it on all test puzzles.

In [ ]:
expression, tokens = react_solve(test_puzzles[0], verbose=True)
print(f"=> {expression} ({tokens:,} tokens)")

In [ ]:
run_method("ReAct", react_solve, test_puzzles)

**4.3 ReAct vs. chain-of-thought.** Compare the accuracy and the tokens per puzzle with CoT. Which mistakes from 3.3 does the environment remove? Look at a transcript of a puzzle that ReAct did not solve (use `verbose=True`; the replies come from the cache): when the numbers are reset, does the model really try a *different* first step? Why does every turn cost more tokens than the previous one? What would the model need in order to search more systematically?

---

*Answer:*

The environment removes all **local** mistakes: wrong arithmetic (we compute the result ourselves), numbers that are not available, wrong `(left: ...)` lists, and false claims of success (we check the last number). Mistakes cost a turn, but they no longer spoil the answer. And the model gets several attempts.

What remains is the **search** problem. After a reset, the model often tries the same or a very similar first step again, because nothing in the conversation tells it which first steps are promising, and keeping track of what it has tried in a growing transcript is hard. The conversation is sent again with every request, so every turn costs more tokens than the previous one: the total grows quadratically with the number of turns. To search systematically, the model would need several candidate steps at every point, an estimate of which ones are promising, and something that keeps track of the open alternatives and backtracks to them. That is what Tree of Thoughts adds, and it moves the bookkeeping out of the model into ordinary code.

---

## 5. Tree of Thoughts

Tree of Thoughts (ToT; Yao et al., 2023b) uses the language model as two components of a classical search algorithm:
- a **thought generator**: given a state, the model *proposes* several possible next steps (a "thought" is one step),
- a **state evaluator**: given a state, the model judges whether the numbers that are left can still reach 24: `sure`, `likely` or `impossible`.

The search itself is ordinary code. We use **breadth-first search** (BFS) with a beam: at every depth, generate the next states of all states in the frontier, evaluate them, and keep only the `breadth` best ones. After three steps, check whether one of the states is 24.

```
depth 0:                     4 5 6 10
                      /          |          \              propose next steps
depth 1:          4 4 5        5 6 6        4 6 15   ...   evaluate: sure / likely / impossible
                    |                                      keep the `breadth` best
depth 2:          4 20   ...
                    |
depth 3:           24 ✓
```

Unlike in ReAct, the model never has to remember which options it has already tried: the search algorithm keeps track of the tree, and the model only answers local questions.

### Generating thoughts

The propose prompt of Yao et al. asks the model to list possible next steps, in our step format:

In [ ]:
PROPOSE_PROMPT = """Use numbers and basic arithmetic operations (+ - * /). Each step, you are only allowed to choose two of the remaining numbers to obtain a new number. Follow the format of the following examples. Do not explain, simply list possible next steps as well as all the remaining numbers and nothing else.

Example: 2 8 8 14
Possible next steps:
2 + 8 = 10 (left: 8 10 14)
8 / 2 = 4 (left: 4 8 14)
14 + 2 = 16 (left: 8 8 16)
2 * 8 = 16 (left: 8 14 16)
8 - 2 = 6 (left: 6 8 14)
14 - 8 = 6 (left: 2 6 8)
14 / 2 = 7 (left: 7 8 8)
14 - 2 = 12 (left: 8 8 12)

Example: 1 3
Possible next steps:
1 + 3 = 4 (left: 4)
1 * 3 = 3 (left: 3)
3 - 1 = 2 (left: 2)
3 / 1 = 3 (left: 3)
1 - 3 = -2 (left: -2)

Input: {input}
Possible next steps:
"""

**5.1 Complete `parse_proposals`**, which turns the model's reply into a list of `(step, new_state)` pairs. Keep only the lines that are valid steps (use `apply_step`, and skip the lines for which it raises a `ValueError`), and skip steps that lead to a state you already have (compare `show(new_state)`). Thanks to `apply_step`, a step the model got wrong, or a wrong `(left: ...)`, cannot enter the search.

In [ ]:
def parse_proposals(reply, state):
    # The valid steps in the reply, as (step, new_state) pairs, without duplicate states
    proposals = []
    seen = set()
    for line in reply.splitlines():
        try:
            new_state = apply_step(state, line)
        except ValueError:
            continue
        if show(new_state) not in seen:
            seen.add(show(new_state))
            proposals.append((line.strip(), new_state))
    return proposals


def propose(state):
    # Ask the model for possible next steps from state; returns (proposals, tokens used)
    reply, tokens = chat([{"role": "user", "content": PROPOSE_PROMPT.format(input=show(state))}], max_tokens=300)
    return parse_proposals(reply, state), tokens

In [ ]:
# ✅ Check parse_proposals
fake_reply = """Possible next steps:
4 + 5 = 9 (left: 6 9 10)
10 - 6 = 4 (left: 4 4 5)
6 - 10 = -4 (left: -4 4 5)
3 * 8 = 24 (left: 24)
6 * 4 = 24 (left: 5 10 24)
4 + 5 = 9 (left: 6 9 10)
That is all."""
proposals = parse_proposals(fake_reply, initial_state("4 5 6 10"))
assert [show(s) for _, s in proposals] == ["6 9 10", "4 4 5", "-4 4 5", "5 10 24"], "keep the valid steps, in order, without duplicates"
assert all(isinstance(step, str) for step, _ in proposals)
print("Looks good ✅")

In [ ]:
proposals, tokens = propose(initial_state(test_puzzles[0]))
print(f"{len(proposals)} valid proposals for {test_puzzles[0]} ({tokens} tokens):")
for step, _ in proposals:
    print("  ", step)

### Evaluating states

The value prompt of Yao et al. asks the model whether the numbers that are left can still reach 24. The demonstrations show it how to reason briefly: try a few combinations, and recognise numbers that are "too big" or "too small". The verdict is on the last line.

In [ ]:
VALUE_PROMPT = """Evaluate if given numbers can reach 24 (sure/likely/impossible). Follow the format of the examples and end with one of the three words on its own line.

Input: 10 14
10 + 14 = 24
sure

Input: 11 12
11 + 12 = 23
12 - 11 = 1
11 * 12 = 132
11 / 12 = 0.91
impossible

Input: 4 4 10
4 + 4 + 10 = 8 + 10 = 18
4 * 10 - 4 = 40 - 4 = 36
(10 - 4) * 4 = 6 * 4 = 24
sure

Input: 4 9 11
9 + 11 + 4 = 20 + 4 = 24
sure

Input: 5 7 8
5 + 7 + 8 = 12 + 8 = 20
(8 - 5) * 7 = 3 * 7 = 21
I cannot obtain 24 now, but numbers are within a reasonable range
likely

Input: 5 6 6
5 + 6 + 6 = 17
(6 - 5) * 6 = 1 * 6 = 6
I cannot obtain 24 now, but numbers are within a reasonable range
likely

Input: 10 10 11
10 + 10 + 11 = 31
(11 - 10) * 10 = 10
10 10 11 are all too big
impossible

Input: 1 3 3
1 * 3 * 3 = 9
(1 + 3) * 3 = 12
1 3 3 are all too small
impossible

Input: {input}
"""

ToT turns the verdicts into scores: `sure` = 20, `likely` = 1 and `impossible` = 0.001. A state can be evaluated several times (with sampling, `n_evaluate > 1`), and then the scores are added up, which averages out some of the model's noise.

**5.2 Complete `parse_value`** for a list of replies. Use the last non-empty line of every reply, ignore upper and lower case, and give a reply without a verdict a score of 0.

In [ ]:
VALUE_SCORES = {"sure": 20.0, "likely": 1.0, "impossible": 0.001}


def parse_value(replies):
    # The total score of a list of evaluation replies
    total = 0.0
    for reply in replies:
        lines = [line.strip().lower() for line in reply.splitlines() if line.strip()]
        if not lines:
            continue
        for word, score in VALUE_SCORES.items():
            if word in lines[-1]:
                total += score
                break
    return total


def value(state, n_evaluate=1):
    # Ask the model n_evaluate times how promising state is; returns (score, tokens used)
    replies, tokens = [], 0
    for sample in range(n_evaluate):
        reply, used = chat(
            [{"role": "user", "content": VALUE_PROMPT.format(input=show(state))}],
            temperature=0.7 if n_evaluate > 1 else 0.0,
            max_tokens=200,
            sample=sample,
        )
        replies.append(reply)
        tokens += used
    return parse_value(replies), tokens

In [ ]:
# ✅ Check parse_value
assert parse_value(["10 + 14 = 24\nsure"]) == 20
assert parse_value(["11 + 12 = 23\n12 - 11 = 1\nimpossible\n\n"]) == 0.001, "ignore empty lines at the end"
assert parse_value(["sure", "likely", "Likely"]) == 22, "add up the scores, and ignore upper and lower case"
assert parse_value(["This is sure to work.\nLet me think more."]) == 0, "only the last line counts"
assert parse_value([]) == 0
print("Looks good ✅")

In [ ]:
for step, new_state in proposals:
    score, tokens = value(new_state)
    print(f"{show(new_state):12} score {score:6.3f}   {'solvable' if solve(new_state) else 'no solution'}   ({tokens} tokens)")

### The search

**5.3 Complete `tot_bfs`.** At every depth:
1. **Generate**: propose the next states of every state in the frontier, and collect them without duplicates (a dictionary from `show(state)` to the state).
2. At depth 3, every candidate has one number left: return its expression if it is 24 (no evaluation needed). If none is, return `None`.
3. **Evaluate** every candidate, and keep the `breadth` candidates with the highest scores as the new frontier.

Add up the tokens of all requests. The functions that propose and evaluate are arguments, so that we can test the search without the API, with a perfect proposer and evaluator.

In [ ]:
def tot_bfs(puzzle, breadth=3, propose_fn=propose, value_fn=value, verbose=False):
    # Tree of Thoughts with breadth-first search; returns (expression or None, tokens used)
    frontier = [initial_state(puzzle)]
    tokens = 0
    for depth in range(1, 4):
        # 1. Generate: the proposals of every state in the frontier, without duplicate states
        candidates = {}  # show(state) -> state
        for state in frontier:
            proposals, used = propose_fn(state)
            tokens += used
            for _, new_state in proposals:
                candidates.setdefault(show(new_state), new_state)
        # 2. At depth 3, one number is left: return the expression of a 24 if there is one, else None
        if depth == 3:
            for state in candidates.values():
                if len(state) == 1 and state[0][0] == 24:
                    return state[0][1], tokens
            return None, tokens
        # 3. Evaluate the candidates, and keep the `breadth` best as the new frontier
        scored = []  # (score, state) pairs
        for state in candidates.values():
            score, used = value_fn(state)
            tokens += used
            scored.append((score, state))
        frontier = [state for _, state in sorted(scored, key=lambda pair: -pair[0])[:breadth]]
        if verbose:
            print(f"depth {depth}: {len(candidates)} candidates, kept:")
            for score, state in sorted(scored, key=lambda pair: -pair[0])[:breadth]:
                print(f"   {score:7.3f}  {show(state):14} {'solvable' if solve(state) else 'no solution'}")
    return None, tokens

In [ ]:
# ✅ Check tot_bfs, with a perfect proposer and evaluator instead of the API


def all_steps(state):
    return next_states(state), 1  # every possible step; pretend that a request costs 1 token


def perfect_value(state):
    return (20.0 if solve(state) else 0.001), 0


def clueless_value(state):
    return 1.0, 0  # every state looks the same


for puzzle in test_puzzles:
    expression, tokens = tot_bfs(puzzle, breadth=1, propose_fn=all_steps, value_fn=perfect_value)
    assert expression is not None and check_answer(expression, puzzle), f"with a perfect evaluator, breadth 1 is enough to solve {puzzle}"
    assert tokens == 3, "with breadth 1, there is one propose request per depth: add up their tokens"
assert tot_bfs("1 1 1 1", breadth=5, propose_fn=all_steps, value_fn=perfect_value)[0] is None
print("Looks good ✅")

solved = sum(tot_bfs(p, breadth=1, propose_fn=all_steps, value_fn=clueless_value)[0] is not None for p in test_puzzles)
print(f"With breadth 1 and an evaluator that has no clue, the search solves {solved} of {len(test_puzzles)} puzzles")

Now with the language model as proposer and evaluator. First one puzzle, with the search tree printed, then the first `N_TOT` test puzzles. This is the expensive part of the notebook: about 15,000 tokens per puzzle.

In [ ]:
expression, tokens = tot_bfs(test_puzzles[0], verbose=True)
print(f"=> {expression} ({tokens:,} tokens)")

In [ ]:
run_method("ToT", tot_bfs, test_puzzles[:N_TOT])

The cell below prints the search trees of the puzzles that ToT did not solve (from the cache, so it costs nothing). `solvable` marks the states from which 24 can still be reached, according to your `solve`.

In [ ]:
for row in results["ToT"]:
    if not row["correct"]:
        print(f"Puzzle {row['puzzle']}")
        tot_bfs(row["puzzle"], verbose=True)
        print()

**5.4 Where does Tree of Thoughts fail?** For each failure, find the depth at which the search lost the last solvable state. Was it because the **generator** never proposed a solvable state, or because the **evaluator** scored the solvable states lower than others? What would you change: the breadth, the number of evaluations, the prompts, or the model? And what does the check in 5.3 (breadth 1 is enough with a perfect evaluator, but not with a clueless one) tell you about where the "intelligence" of ToT sits?

---

*Answer:*

Both failures occur. The **generator** proposes only five to ten of the 36 possible first steps and, from three numbers, often misses the one step that works. The **evaluator** is the more common culprit: it says `likely` to almost every state with three numbers, so the scores hardly separate good from bad states, and it sometimes calls solvable states `impossible`, in particular when the solution needs fractions (like `8 / (3 - 8 / 3)`).

Possible changes: a larger **breadth** (more states survive, but more requests), several **evaluations** per state with sampling (`n_evaluate = 3` at temperature 0.7, as in the paper, averages out noise), better **prompts** (more or better demonstrations), or a stronger **model**. Each costs more tokens.

The check shows that the search algorithm only *organises* the work: with a perfect evaluator, no search is needed at all (breadth 1 follows the right path), and with a useless evaluator, breadth 1 is a random walk that rarely finds 24. The quality of ToT comes from the model's proposals and evaluations; the search lets you trade more requests for robustness against their mistakes.

---

## 6. Comparison

In [ ]:
print(f"{'method':8} {'solved, first ' + str(N_TOT):>16} {'solved, all ' + str(N_TEST):>15} {'tokens per puzzle':>18}")
for name, rows in results.items():
    first = f"{sum(row['correct'] for row in rows[:N_TOT])}/{N_TOT}"
    every = f"{sum(row['correct'] for row in rows)}/{len(rows)}" if len(rows) == N_TEST else "-"
    tokens = sum(row["tokens"] for row in rows) / len(rows)
    print(f"{name:8} {first:>16} {every:>15} {tokens:18,.0f}")

**6.1 Accuracy against cost.**
1. Which method would you use for the Game of 24, and which for a problem without a way to check steps automatically?
2. ToT uses many times more tokens than CoT. Is the comparison fair? What could CoT achieve with the same budget, for example by sampling many chains and checking each answer with `check_answer`?
3. *Reasoning models* (such as OpenAI's o1, DeepSeek-R1, or this model with its thinking mode on) are trained with reinforcement learning to write very long hidden chains of thought before they answer. How does what happens in such a chain relate to what you built in this notebook?

---

*Answer:*

1. For the Game of 24, where every step and every answer can be checked, ReAct or ToT: the environment catches mistakes, and ToT also searches. Without an automatic check, ReAct's environment has nothing to report, and ToT relies entirely on the model's own evaluations; then CoT (perhaps with self-consistency) is the cheap default, and ToT only pays off if the model's evaluations are good and mistakes are expensive.
2. Not entirely: ToT gets many more requests. A fair comparison gives every method the same budget. Yao et al. (2023b) did this: the best of 100 CoT chains, picked with an oracle that knows the correct answer, solved 49 % of the puzzles, against 74 % for ToT with a similar or smaller budget. So part, but not all, of ToT's advantage comes from simply spending more. With `check_answer` as the oracle, sampling CoT chains until one is correct is a strong and simple baseline, and it works for any problem with a verifier.
3. Reasoning models internalise the search. Their long chains of thought contain proposals ("let me try 10 − 6 …"), evaluations ("that gives 20, which doesn't help"), and backtracking ("wait, let's go back and try …"). They learned this through reinforcement learning with **verifiable rewards**, on exactly the kind of problems where answers can be checked automatically (math, code, puzzles). There is no external search algorithm or environment: the cost moves into many hidden reasoning tokens. The frameworks in this notebook make the same ingredients explicit, so you can inspect and control them.

---

### Going further
- **Turn the thinking on**: copy `chat` with `reasoning_effort="default"` instead of `"none"` and a larger `max_tokens`, and run the IO prompt with it. Compare the accuracy and the tokens (`response.usage`) with ToT.
- **Evaluate with sampling**: run ToT with `n_evaluate=3` (as in the paper; use `value_fn=lambda s: value(s, n_evaluate=3)`) or with `breadth=5`, and see whether the accuracy pays for the extra tokens.
- **Depth-first search**: implement ToT with DFS, which follows the best state first and backtracks when a state is evaluated as `impossible`. How many requests does it need compared with BFS?

Many more frameworks build on these ideas:

| Framework | Core idea | Paper |
|----------|-----------|-------|
| **Few-shot prompting** | Show the model a few solved examples | [Brown et al., 2020](https://arxiv.org/abs/2005.14165) |
| **Chain of Thought (CoT)** | Write out the intermediate steps before the answer | [Wei et al., 2022](https://arxiv.org/abs/2201.11903) |
| **Self-consistency** | Sample several chains of thought and take a majority vote | [Wang et al., 2023](https://arxiv.org/abs/2203.11171) |
| **ReAct** | Interleave reasoning with actions in an environment | [Yao et al., 2023a](https://arxiv.org/abs/2210.03629) |
| **Tree of Thoughts (ToT)** | Search over partial solutions, with the model as generator and evaluator | [Yao et al., 2023b](https://arxiv.org/abs/2305.10601) |
| **Reflexion** | An agent reflects on its failed attempts in words and uses the reflections in the next attempt | [Shinn et al., 2023](https://arxiv.org/abs/2303.11366) |
| **Graph of Thoughts (GoT)** | Reasoning as a graph, in which thoughts can also be merged and refined | [Besta et al., 2024](https://arxiv.org/abs/2308.09687) |
| **Language Agent Tree Search (LATS)** | Monte Carlo tree search with the model as policy, value function and self-reflection | [Zhou et al., 2024](https://arxiv.org/abs/2310.04406) |
| **Fleet of Agents (FoA)** | A fleet of agents searches with genetic-type particle filtering: promising agents are resampled, unpromising ones dropped | [Klein et al., 2024](https://arxiv.org/abs/2405.06691) |

## 7. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 7.1. Chain-of-thought and search

Why does chain-of-thought prompting help little on the Game of 24?

A. The model cannot compute with fractions<br>
B. The chain is generated left to right in a single pass, so a bad early step cannot be undone<br>
C. Chain-of-thought prompts do not fit into the context window<br>
D. The puzzles require knowledge that the model has not seen during training<br>

**Answer:** **B**

A chain of thought commits to its first step. Most puzzles have only a few good first steps, and when the model picks a bad one, the rest of the chain cannot reach 24 however well it reasons. Recovering requires going back and trying another step: search.

---

### 7.2. ReAct

In ReAct, where do the observations come from?

A. The model generates them after each of its actions<br>
B. The environment that executes the action: a tool, an API, or your code<br>
C. A second language model that checks the first one<br>
D. The few-shot demonstrations in the prompt<br>

**Answer:** **B**

The point of ReAct is that the observations are *not* generated by the model: they come from the outside world, so they can correct the model's mistakes and give it information it does not have. If the model wrote its own observations, it could simply hallucinate success.

---

### 7.3. The state evaluator

What is the role of the state evaluator in Tree of Thoughts?

A. It checks whether the final answer is correct<br>
B. It estimates how promising a partial solution is, so that the search can prune unpromising branches<br>
C. It proposes the next steps<br>
D. It reduces the number of tokens per request<br>

**Answer:** **B**

The evaluator works as a heuristic: it scores intermediate states (`sure` / `likely` / `impossible`), and the search keeps only the most promising ones. The final answer is checked separately; proposing is the generator's job (C).

---

### 7.4. The cost of Tree of Thoughts

You run ToT with BFS and breadth 3 on a Game of 24 puzzle, as in this notebook. Every propose request returns 8 new, distinct states, and every state at depths 1 and 2 is evaluated once. How many requests does the puzzle take?

A. 7<br>
B. 32<br>
C. 39<br>
D. 192<br>

**Answer:** **C**

Propose requests: 1 at depth 1, 3 at depth 2 (one per state in the frontier), and 3 at depth 3: 7 in total. Evaluations: 8 states at depth 1, and 3 × 8 = 24 at depth 2: 32 in total. States at depth 3 are checked without the model. 7 + 32 = 39 requests, against 1 for chain-of-thought.

---

### 7.5. A perfect evaluator

Suppose the evaluator is perfect (it always knows whether 24 can still be reached from a state), and the generator proposes every possible step. What is the smallest breadth for which ToT with BFS solves every puzzle that has a solution?

A. 1<br>
B. 3<br>
C. 5<br>
D. 36<br>

**Answer:** **A**

With a perfect evaluator, a solvable state always scores higher than an unsolvable one. From a solvable state, at least one next state is solvable, so keeping only the best state at every depth leads straight to 24. A wider beam is only needed because real evaluators make mistakes.